## Statistical Inference

In this section, we move beyond and apply statistical inference to evaluate whether the patterns\
observed in stock returns are likely to reflect real underlying effects rather than random variation.

### Target Population

The target population in this analysis is the distribution of daily stock returns for the selected\
stocks (AAPL, MSFT, AMZN, GOOGL, META) over the studied period. Since we have only historical daily\
returns from 2010 onward, our dataset is treated as a sample from the broader process that generates\
stock returns over time.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats

In [5]:
data_path = Path("../02_magic/01_data/raw/stock_prices.csv")
df = pd.read_csv(data_path)

print("Shape:", df.shape)
df.head()

Shape: (19836, 8)


,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2010-01-04,7.622500,7.660714,7.585000,7.643214,6.412384,493729600,AAPL
1,2010-01-05,7.664286,7.699643,7.616071,7.656429,6.423470,601904800,AAPL
2,2010-01-06,7.656429,7.686786,7.526786,7.534643,6.321297,552160000,AAPL
3,2010-01-07,7.562500,7.571429,7.466071,7.520714,6.309608,477131200,AAPL
4,2010-01-08,7.510714,7.571429,7.466429,7.570714,6.351556,447610800,AAPL


In [9]:
df_inf = df.copy()

df_inf["Date"] = pd.to_datetime(df_inf["Date"])
df_inf["Return_1d"] = df_inf.groupby("Ticker")["Close"].pct_change()

df_inf = df_inf.dropna(subset=["Return_1d"]).copy()

alpha = 0.05

df_inf[["Date", "Ticker", "Close", "Return_1d"]].head()

,Date,Ticker,Close,Return_1d
1,2010-01-05,AAPL,7.656429,0.001729
2,2010-01-06,AAPL,7.534643,-0.015906
3,2010-01-07,AAPL,7.520714,-0.001849
4,2010-01-08,AAPL,7.570714,0.006648
5,2010-01-11,AAPL,7.503929,-0.008821


### Confidence Intervals for Mean Daily Returns

As a first step, I calculate 95% confidence intervals for the mean daily return of each stock.\
This shows the range in which the true average return is likely to fall based on the observed data.

In [15]:
ci_results = []

for ticker in df_inf["Ticker"].unique():
    sample = df_inf.loc[df_inf["Ticker"] == ticker, "Return_1d"].dropna()
    
    mean_ret = sample.mean()
    sem = stats.sem(sample)
    n = len(sample)

    ci_low, ci_high = stats.t.interval(
        confidence=0.95,
        df=n - 1,
        loc=mean_ret,
        scale=sem
    )

    ci_results.append({
        "Ticker": ticker,
        "Mean_Return": mean_ret,
        "CI_95_Lower": ci_low,
        "CI_95_Upper": ci_high
    })

ci_df = pd.DataFrame(ci_results).sort_values("Mean_Return", ascending=False)
ci_df.round(6)

,Ticker,Mean_Return,CI_95_Lower,CI_95_Upper
3,META,0.001090,0.000259,0.001922
1,AMZN,0.001056,0.000422,0.001690
0,AAPL,0.001017,0.000473,0.001560
2,GOOGL,0.000870,0.000336,0.001404
4,MSFT,0.000740,0.000245,0.001236


### Confidence Interval Results

All five stocks have 95% confidence intervals above zero, which suggests their average\
daily returns were positive over the observed period. **META** and **AMZN** show the highest\
mean returns, while **MSFT** has the lowest, although the differences are still small.

### Hypothesis 1: Mean Daily Return vs 0

Next, I test whether the average daily return of each stock is different from 0 using a one-sample t-test.

- **Null hypothesis ($H_0$):** the mean daily return is equal to 0  
- **Alternative hypothesis ($H_1$):** the mean daily return is different from 0  

The significance level is set to 0.05.

In [11]:
ttest_zero_results = []

for ticker in df_inf["Ticker"].unique():
    sample = df_inf.loc[df_inf["Ticker"] == ticker, "Return_1d"].dropna()
    t_stat, p_value = stats.ttest_1samp(sample, popmean=0)

    ttest_zero_results.append({
        "Ticker": ticker,
        "Mean_Return": sample.mean(),
        "T_Statistic": t_stat,
        "P_Value": p_value,
        "Reject_H0_at_5pct": p_value < alpha
    })

ttest_zero_df = pd.DataFrame(ttest_zero_results).sort_values("P_Value")
ttest_zero_df.round(6)

,Ticker,Mean_Return,T_Statistic,P_Value,Reject_H0_at_5pct
0,AAPL,0.001017,3.666937,0.000249,True
1,AMZN,0.001056,3.267004,0.001096,True
2,GOOGL,0.000870,3.193135,0.001418,True
4,MSFT,0.000740,2.928407,0.003426,True
3,META,0.001090,2.571674,0.010162,True


### Hypothesis 1 Results

For all five stocks, the p-values are below 0.05, so the **null hypothesis is rejected** in each case.\
This means daily return of every stock is statistically different from 0 over the observed period.\
Average returns are still quite small, so the result is statistically meaningful but modest in size.

### Hypothesis 2: AAPL vs MSFT Mean Returns

Here, I compare whether **AAPL** and **MSFT** have the same average daily return using a two-sample t-test.

- **Null hypothesis ($H_0$):** AAPL and MSFT have the same mean daily return  
- **Alternative hypothesis ($H_1$):** AAPL and MSFT have different mean daily returns  

The significance level is set to 0.05.

In [12]:
stock_1 = "AAPL"
stock_2 = "MSFT"

sample_1 = df_inf.loc[df_inf["Ticker"] == stock_1, "Return_1d"].dropna()
sample_2 = df_inf.loc[df_inf["Ticker"] == stock_2, "Return_1d"].dropna()

t_stat, p_value = stats.ttest_ind(sample_1, sample_2, equal_var=False)

stock_comparison_df = pd.DataFrame({
    "Stock_1": [stock_1],
    "Stock_2": [stock_2],
    "Mean_1": [sample_1.mean()],
    "Mean_2": [sample_2.mean()],
    "T_Statistic": [t_stat],
    "P_Value": [p_value],
    "Reject_H0_at_5pct": [p_value < alpha]
})

stock_comparison_df.round(6)

,Stock_1,Stock_2,Mean_1,Mean_2,T_Statistic,P_Value,Reject_H0_at_5pct
0,AAPL,MSFT,0.001017,0.00074,0.736574,0.461403,False


### Hypothesis 2 Results

The p-value is **above** 0.05, so the null hypothesis is not rejected.\
The difference between the average daily returns of **AAPL** and **MSFT** **is not statistically significant** in this sample.\
Although **AAPL** has a slightly higher mean return, the gap is too small to conclude that the two stocks behave differently on average.

### Hypothesis 3: Crisis vs Non-Crisis Returns

Here, I compare average daily returns during the 2020 crash period and the rest of the sample using a two-sample t-test.

- **Crisis period:** 2020-02-20 to 2020-04-30  
- **Non-crisis period:** all other dates  
- **Null hypothesis ($H_0$):** the mean daily return is the same in both periods  
- **Alternative hypothesis ($H_1$):** the mean daily return is different  

The significance level is set to 0.05.

In [13]:
crisis_start = "2020-02-20"
crisis_end = "2020-04-30"

df_inf["Period"] = np.where(
    (df_inf["Date"] >= crisis_start) & (df_inf["Date"] <= crisis_end),
    "Crisis",
    "Non-Crisis"
)

crisis_returns = df_inf.loc[df_inf["Period"] == "Crisis", "Return_1d"].dropna()
non_crisis_returns = df_inf.loc[df_inf["Period"] == "Non-Crisis", "Return_1d"].dropna()

t_stat, p_value = stats.ttest_ind(crisis_returns, non_crisis_returns, equal_var=False)

period_comparison_df = pd.DataFrame({
    "Crisis_Mean_Return": [crisis_returns.mean()],
    "Non_Crisis_Mean_Return": [non_crisis_returns.mean()],
    "T_Statistic": [t_stat],
    "P_Value": [p_value],
    "Reject_H0_at_5pct": [p_value < alpha]
})

period_comparison_df.round(6)

,Crisis_Mean_Return,Non_Crisis_Mean_Return,T_Statistic,P_Value,Reject_H0_at_5pct
0,0.000218,0.00096,-0.262321,0.79329,False


### Hypothesis 3 Results

The p-value is well above 0.05, so the **null hypothesis is not rejected**.\
This means the average daily return during the 2020 crash period was not statistically different\
from the rest of the sample in this test. Although the crisis-period mean return is lower, the difference\
is too small relative to the variability in returns to be considered significant.